# Exploration de l'ordre d'imputation

Ce notebook explore comment `HighFrequencyImputer` decide, a l'ajustement, de **la cascade de frequences** et de **l'ordre des variables a l'interieur d'une etape**. Trois surfaces ajustees portent cette decision :

| Attribut | Contenu |
|---|---|
| `frequency_progression_` | liste ordonnee des frequences d'etape (la cible en dernier) |
| `variable_categories_` | cles de variables par categorie `'aggregate'` / `'impute'` / `'target_freq'`, classees **par couple (entite, colonne)** sur un panel |
| `imputation_order_` | `{label d'etape: [colonnes ordonnees]}` : **peuple seulement sous `covariate_strategy='model'`** (le seul mode ou l'ordre change une valeur) |
| `imputation_cv_scores_` | `{label d'etape: {colonne: score}}` : en plus, sous `fit_predict_order='cv'` |

Le composant `VariableOrderer` (reexporte) porte la logique de rang ; l'imputeur l'instancie et l'interroge par etape.

Trois scenarios :

1. **Series temporelles simples** : frequences mixtes, pas de dimension de panel
2. **Panel homogene** : memes frequences par indicateur pour toutes les entites
3. **Panel heterogene** : frequences differentes par entite

Pour rendre la cascade et l'ordre observables, les imputeurs sont construits avec `covariate_strategy='model'` et `impute_intermediate_frequencies='covariates_only'` (etapes intermediaires qui enrichissent les covariables sans bruiter la cible).

---

## 0. Imports & utilitaires

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LinearRegression

from tsforecast.frequency import HighFrequencyImputer, VariableOrderer

In [ ]:
# Palette de couleurs par frequence
FREQ_COLORS = {
    "A": "#e15759",   # Annuelle      -> rouge
    "Y": "#e15759",
    "Q": "#f28e2b",   # Trimestrielle -> orange
    "M": "#4e79a7",   # Mensuelle     -> bleu
}

CAT_COLORS = {
    "impute": "#59a14f",       # Vert   -> a imputer
    "aggregate": "#f28e2b",    # Orange -> a agreger
    "target_freq": "#4e79a7",  # Bleu   -> deja a la frequence cible
}

FREQ_LABELS = {"A": "Annuelle", "Y": "Annuelle", "Q": "Trimestrielle", "M": "Mensuelle"}
CAT_LABELS = {"impute": "A imputer", "aggregate": "A agreger", "target_freq": "Frequence cible"}


def normalize_freq_label(freq: str) -> str:
    """Normalize a frequency code to a single-letter display label."""
    mapping = {"MS": "M", "QS": "Q", "AS": "A", "YS": "A", "Y": "A"}
    return mapping.get(freq, freq)


def print_section(title: str) -> None:
    """Print a formatted section header."""
    print("\n" + "=" * 64)
    print(f"  {title}")
    print("=" * 64)


def key_category(imputer, key, default="unknown"):
    """Look up a variable key's category in ``variable_categories_``."""
    for cat, keys in imputer.variable_categories_.items():
        if key in keys:
            return cat
    return default


def stage_labels(imputer) -> list:
    """Return the ordered stage labels of a fitted imputer."""
    return [
        imputer._stage_frequency_label(stage_freq)
        for stage_freq in imputer.frequency_progression_
    ]

In [ ]:
def summarize_stage_order(imputer: HighFrequencyImputer) -> pd.DataFrame:
    """Flatten ``imputation_order_`` into one row per (stage, rank, column).

    Args:
        imputer: Fitted HighFrequencyImputer.

    Returns:
        DataFrame with columns: etape, rang, colonne, score_cv (NaN when the
        stage fell back to a frequency order).
    """
    rows = []
    cv_scores = getattr(imputer, "imputation_cv_scores_", {}) or {}
    for stage, columns in imputer.imputation_order_.items():
        stage_scores = cv_scores.get(stage, {})
        for rank, column in enumerate(columns, start=1):
            rows.append({
                "etape": stage,
                "rang": rank,
                "colonne": column,
                "score_cv": stage_scores.get(column, np.nan),
            })
    return pd.DataFrame(rows)


def plot_frequency_progression(imputer: HighFrequencyImputer, title: str) -> None:
    """Draw the stage cascade as a left-to-right arrow of colored blocks."""
    labels = stage_labels(imputer)
    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 2.2), 1.8))
    for i, label in enumerate(labels):
        disp = normalize_freq_label(label)
        ax.add_patch(mpatches.FancyBboxPatch(
            (i, 0), 0.8, 1, boxstyle="round,pad=0.02",
            facecolor=FREQ_COLORS.get(disp, "#aaaaaa"), edgecolor="white"))
        ax.text(i + 0.4, 0.5, label, ha="center", va="center",
                color="white", fontweight="bold")
        if i < len(labels) - 1:
            ax.annotate("", xy=(i + 1, 0.5), xytext=(i + 0.8, 0.5),
                        arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.set_xlim(-0.2, len(labels))
    ax.set_ylim(-0.2, 1.2)
    ax.axis("off")
    ax.set_title(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_stage_order(imputer: HighFrequencyImputer, title: str, figsize=(14, 4)) -> None:
    """Bar chart of the per-stage variable order (one subplot per stage)."""
    order = imputer.imputation_order_
    if not order:
        print("imputation_order_ vide : construire l'imputeur avec "
              "covariate_strategy='model' pour peupler l'ordre.")
        return
    fig, axes = plt.subplots(1, len(order), figsize=figsize, squeeze=False)
    fig.suptitle(title, fontsize=13, fontweight="bold")
    for ax, (stage, columns) in zip(axes[0], order.items()):
        disp = normalize_freq_label(stage)
        ax.barh(range(len(columns)), [len(columns) - i for i in range(len(columns))],
                color=FREQ_COLORS.get(disp, "#aaaaaa"), edgecolor="white", height=0.7)
        ax.set_yticks(range(len(columns)))
        ax.set_yticklabels(columns, fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel("Priorite ->")
        ax.set_title(f"Etape {stage}", fontsize=11)
        for i in range(len(columns)):
            ax.text(len(columns) - i - 0.1, i, f"#{i + 1}", va="center", ha="right",
                    color="white", fontweight="bold", fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_freq_heatmap_panel(imputer: HighFrequencyImputer, title: str) -> None:
    """Heatmap of the detected frequency per (entity, variable) on a panel."""
    records = {}
    for key, freq in imputer.detected_frequencies_.items():
        if isinstance(key, tuple):
            entity = str(key[:-1][0] if len(key) == 2 else key[:-1])
            variable = key[-1]
        else:
            entity, variable = "-", key
        records[(entity, variable)] = normalize_freq_label(freq)

    df_heat = pd.Series(records).unstack(level=1)
    freq_num = {f: i for i, f in enumerate(["A", "Q", "M"])}
    num_map = df_heat.apply(lambda col: col.map(lambda x: freq_num.get(x, np.nan)))

    fig, ax = plt.subplots(figsize=(max(8, len(df_heat.columns) * 1.2),
                                    max(4, len(df_heat) * 0.6)))
    cmap = plt.get_cmap("RdYlBu", 3)
    ax.imshow(num_map.values, cmap=cmap, vmin=0, vmax=2, aspect="auto")
    ax.set_xticks(range(len(df_heat.columns)))
    ax.set_xticklabels(df_heat.columns, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(df_heat.index)))
    ax.set_yticklabels(df_heat.index, fontsize=9)
    for i in range(len(df_heat.index)):
        for j in range(len(df_heat.columns)):
            val = df_heat.values[i, j]
            if pd.notna(val):
                ax.text(j, i, val, ha="center", va="center", fontsize=9,
                        fontweight="bold", color="white")
    legend_patches = [
        mpatches.Patch(color=cmap(0.0), label="Annuelle (A)"),
        mpatches.Patch(color=cmap(0.5), label="Trimestrielle (Q)"),
        mpatches.Patch(color=cmap(1.0), label="Mensuelle (M)"),
    ]
    ax.legend(handles=legend_patches, loc="upper right",
              bbox_to_anchor=(1.25, 1), fontsize=8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Variables")
    ax.set_ylabel("Entites")
    plt.tight_layout()
    plt.show()

---
## 1. Series temporelles simples a frequences mixtes

Un seul << panel >> implicite (pas d'entite), des variables annuelles, trimestrielles et mensuelles. Frequence cible : **mensuelle**.

**Attendu** : la frequence source **la plus basse n'ouvre jamais d'etape** (rien n'y est imputable). La cascade part donc de la deuxieme frequence la plus basse et monte jusqu'a la cible. Une variable annuelle est imputee des la premiere etape et re-predite a chaque etape suivante, sur toute sa periode (les ancres sont toujours desagregees).

In [ ]:
dates_m = pd.date_range("2018-01-01", periods=60, freq="MS")
rng = np.random.default_rng(seed=42)

df_ts = pd.DataFrame(index=dates_m)
df_ts.index.name = "date"

# Variables mensuelles - deja a la frequence cible
df_ts["var_m_1"] = rng.normal(100, 10, 60).cumsum() / 60 + 50
df_ts["var_m_2"] = rng.normal(0, 5, 60).cumsum() + 200
df_ts["var_m_3"] = rng.normal(0, 3, 60).cumsum() + 80

# Variables trimestrielles - observees au seul premier mois de chaque trimestre
dates_q = pd.date_range("2018-01-01", periods=20, freq="QS")
for name, scale in [("var_q_1", 200), ("var_q_2", 50), ("var_q_3", 300)]:
    s_q = pd.Series(rng.normal(scale, scale * 0.05, 20), index=dates_q)
    df_ts[name] = s_q.reindex(dates_m)

# Variables annuelles - observees au seul mois de janvier
dates_a = pd.date_range("2018-01-01", periods=5, freq="YS")
for name, scale in [("var_a_1", 1000), ("var_a_2", 500)]:
    s_a = pd.Series(rng.normal(scale, scale * 0.05, 5), index=dates_a)
    df_ts[name] = s_a.reindex(dates_m)

print(f"Dimensions : {df_ts.shape}")
print(df_ts.isna().sum().to_string())
df_ts.head(6)

In [ ]:
imputer_ts = HighFrequencyImputer(
    target_frequency="M",
    estimator=LinearRegression(),
    covariate_strategy="model",
    impute_intermediate_frequencies="covariates_only",
    fit_predict_order="frequency",
    imputation_scope="strict",
)
imputer_ts.fit(df_ts)

In [ ]:
print_section("Frequences detectees")
for col, freq in imputer_ts.detected_frequencies_.items():
    cat = key_category(imputer_ts, col, default="?")
    print(f"  {col:<12} -> {normalize_freq_label(freq):<3}  [{CAT_LABELS.get(cat, cat)}]")

print_section("Cascade de frequences (frequency_progression_)")
print("  " + "  ->  ".join(stage_labels(imputer_ts)))

print_section("Ordre d'imputation par etape (imputation_order_)")
for stage, columns in imputer_ts.imputation_order_.items():
    print(f"  etape {stage} : {columns}")

In [ ]:
plot_frequency_progression(imputer_ts, "Scenario 1 - cascade de frequences (cible : M)")
display(summarize_stage_order(imputer_ts))
plot_stage_order(imputer_ts, "Scenario 1 - ordre des variables par etape")

### Observations

- Les variables sources sont annuelles et trimestrielles, la cible mensuelle : la cascade `frequency_progression_` est `Q -> M`. L'annuel, frequence la plus basse, n'ouvre pas d'etape.
- `var_a_*` (annuelles) sont donc imputees **des l'etape `Q`**, puis re-predites a l'etape `M` : elles apparaissent dans les deux `imputation_order_`.
- `var_m_*` sont classees `target_freq` : elles n'apparaissent dans aucun `imputation_order_`.
- Sous `fit_predict_order='frequency'`, l'ordre intra-etape suit la frequence source (la plus basse d'abord) ; `imputation_cv_scores_` reste vide.

---

## 2. Panel homogene

Trois entites (`region_A/B/C`) avec exactement **les memes variables aux memes frequences**. Frequence cible : **mensuelle**.

**Attendu** : la classification se fait par couple (entite, colonne), mais comme toutes les entites partagent les memes frequences, chaque etape a un label de frequence simple et le meme jeu de colonnes que le cas serie simple, repete pour les trois entites.

In [ ]:
def build_entity_block(entity, dates_m, rng, var_specs):
    """Build a single-entity mixed-frequency block with a (entity, date) index."""
    freq_map = {"M": "MS", "Q": "QS", "A": "YS"}
    n_periods = {"M": len(dates_m), "Q": len(dates_m) // 3 + 1, "A": len(dates_m) // 12 + 1}
    df = pd.DataFrame(index=dates_m)
    df.index.name = "date"
    for var, (freq, base) in var_specs.items():
        ref = pd.date_range(dates_m[0], periods=n_periods[freq], freq=freq_map[freq])
        df[var] = pd.Series(rng.normal(base, base * 0.05, len(ref)), index=ref).reindex(dates_m)
    df["entity"] = entity
    return df.reset_index().set_index(["entity", "date"])


dates_m = pd.date_range("2018-01-01", periods=60, freq="MS")
rng = np.random.default_rng(seed=0)

var_specs_homo = {
    "prod_indus": ("M", 100), "emploi": ("M", 500), "export": ("M", 200),
    "pib": ("Q", 1000), "invest": ("Q", 300), "pop_active": ("A", 5000),
}
entities_homo = ["region_A", "region_B", "region_C"]
df_panel_homo = pd.concat([build_entity_block(e, dates_m, rng, var_specs_homo)
                           for e in entities_homo])
print(f"Dimensions : {df_panel_homo.shape}")
df_panel_homo.head(6)

In [ ]:
imputer_homo = HighFrequencyImputer(
    target_frequency="M",
    estimator=LinearRegression(),
    covariate_strategy="model",
    impute_intermediate_frequencies="covariates_only",
    fit_predict_order="frequency",
    imputation_scope="strict",
)
imputer_homo.fit(df_panel_homo)

In [ ]:
print_section("Cascade de frequences (panel homogene)")
print("  " + "  ->  ".join(stage_labels(imputer_homo)))

print_section("Categories par couple (entite, colonne)")
for cat, keys in imputer_homo.variable_categories_.items():
    print(f"  {CAT_LABELS.get(cat, cat):<15} : {len(keys)} couples")

print_section("Ordre d'imputation par etape")
for stage, columns in imputer_homo.imputation_order_.items():
    print(f"  etape {stage} : {columns}")

In [ ]:
plot_freq_heatmap_panel(imputer_homo, "Panel homogene - frequence par entite x variable")
plot_frequency_progression(imputer_homo, "Scenario 2 - cascade de frequences (cible : M)")
display(summarize_stage_order(imputer_homo))
plot_stage_order(imputer_homo, "Scenario 2 - ordre des variables par etape")

### Observations

- Toutes les entites partagent leurs frequences : chaque etape porte un label simple (`Q`, `M`).
- `variable_categories_` compte des **couples** : `pop_active` annuelle pour trois entites = trois cles `impute` ; `pib` / `invest` trimestrielles idem.
- A l'etape `Q`, seul `pop_active` (annuel) est imputable : une seule colonne, l'ordonnanceur court-circuite et `imputation_order_` ne porte pas d'entree `Q`.
- A l'etape `M`, `pop_active`, `invest` et `pib` sont toutes imputables : elles y sont ordonnees une seule fois, quel que soit le nombre d'entites (un modele mutualise par couple (colonne, etape)).

---

## 3. Panel heterogene - frequences differentes par entite

Trois profils asymetriques :

| Entite | Profil |
|--------|--------|
| `pays_nord` | beaucoup de variables basse frequence |
| `pays_sud`  | profil equilibre |
| `pays_est`  | beaucoup de variables haute frequence |

**Attendu** : une meme colonne peut etre annuelle pour une entite et la frequence cible pour une autre. Elle porte alors des categories differentes selon l'entite. La cible etant trimestrielle et aucune variable annuelle n'ouvrant d'etape, la cascade se reduit a l'unique etape `Q`.

In [ ]:
rng = np.random.default_rng(seed=7)

var_specs_nord = {
    "prod_indus": ("A", 100), "emploi": ("A", 500), "export": ("Q", 200),
    "pib": ("A", 1000), "invest": ("Q", 300), "pop_active": ("A", 5000),
    "consomm": ("Q", 800),
}
var_specs_sud = {
    "prod_indus": ("M", 110), "emploi": ("M", 520), "export": ("M", 210),
    "pib": ("Q", 1050), "invest": ("Q", 310), "pop_active": ("A", 4800),
    "tx_chomage": ("Q", 8),
}
var_specs_est = {
    "prod_indus": ("M", 95), "emploi": ("M", 480), "export": ("M", 195),
    "pib": ("Q", 980), "invest": ("M", 290), "pop_active": ("A", 5100),
    "ipc": ("M", 102),
}
df_panel_het = pd.concat([
    build_entity_block("pays_nord", dates_m, rng, var_specs_nord),
    build_entity_block("pays_sud", dates_m, rng, var_specs_sud),
    build_entity_block("pays_est", dates_m, rng, var_specs_est),
])
print(f"Dimensions : {df_panel_het.shape}")
df_panel_het.head(6)

In [ ]:
imputer_het = HighFrequencyImputer(
    target_frequency="Q",
    estimator=LinearRegression(),
    covariate_strategy="model",
    impute_intermediate_frequencies="covariates_only",
    fit_predict_order="frequency",
    imputation_scope="strict",
)
imputer_het.fit(df_panel_het)

In [ ]:
print_section("Frequences detectees (panel heterogene)")
for key, freq in sorted(imputer_het.detected_frequencies_.items(), key=lambda x: str(x[0])):
    cat = key_category(imputer_het, key, default="?")
    print(f"  {str(key):<32} -> {normalize_freq_label(freq):<3}  [{CAT_LABELS.get(cat, cat)}]")

print_section("Cascade de frequences (labels d'etape)")
for stage_freq, label in zip(imputer_het.frequency_progression_, stage_labels(imputer_het)):
    print(f"  {label:<10} <- {stage_freq}")

print_section("Ordre d'imputation par etape")
for stage, columns in imputer_het.imputation_order_.items():
    print(f"  etape {stage} : {columns}")

In [ ]:
plot_freq_heatmap_panel(imputer_het, "Panel heterogene - frequence par entite x variable")
plot_frequency_progression(imputer_het, "Scenario 3 - cascade de frequences (cible : Q)")
display(summarize_stage_order(imputer_het))
plot_stage_order(imputer_het, "Scenario 3 - ordre des variables par etape", figsize=(15, 5))

In [ ]:
# Vue "plan" : les etapes de imputation_plan_, regroupees par frequence d'etape
print_section("imputation_plan_.by_stage()")
for stage_label, steps in imputer_het.imputation_plan_.by_stage().items():
    print(f"\n  etape {stage_label} - {len(steps)} pas")
    for step in steps:
        var = step.var_key[-1] if isinstance(step.var_key, tuple) else step.var_key
        print(f"    {str(var):<14} fallback={step.is_fallback}  n_features={len(step.feature_cols)}")

### Observations

- `pib` est annuelle pour `pays_nord` mais trimestrielle (la cible) pour `pays_sud` / `pays_est` : cote nord elle est `impute` a l'etape `Q`, cote sud/est elle est `target_freq`. La meme colonne porte donc deux categories selon l'entite.
- `invest` est mensuelle pour `pays_est` (plus haute que la cible `Q`) : elle y est classee `aggregate`, pas `impute`.
- Le label d'etape reste simple ici car toutes les entites traversant l'etape `Q` la partagent ; si elles divergeaient, `_stage_frequency_label` produirait un label composite listant la frequence de chaque entite.
- `imputation_plan_.by_stage()` donne la vue executee : un pas par (colonne, etape). Sur ce petit panel heterogene, `covariate_strategy='model'` ne trouve pas de jeu d'entrainement exploitable pour les ancres annuelles et chaque pas bascule en repli (`is_fallback=True`, `n_features=0`) : l'ordre reste calcule, mais il ne change aucune valeur.

---

## 4. Recapitulatif comparatif

In [ ]:
def format_progression(imputer, name):
    """Return a formatted cascade + per-stage order summary."""
    lines = [f"-- {name} --", "  cascade : " + "  ->  ".join(stage_labels(imputer))]
    for stage, columns in imputer.imputation_order_.items():
        lines.append(f"  etape {stage:<10} : {columns}")
    return "\n".join(lines)

print(format_progression(imputer_ts, "Scenario 1 - series simples (cible M)"))
print()
print(format_progression(imputer_homo, "Scenario 2 - panel homogene (cible M)"))
print()
print(format_progression(imputer_het, "Scenario 3 - panel heterogene (cible Q)"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), squeeze=False)
scenarios = [
    (imputer_ts, "Scenario 1\nSeries simples"),
    (imputer_homo, "Scenario 2\nPanel homogene"),
    (imputer_het, "Scenario 3\nPanel heterogene"),
]
for ax, (imputer, title) in zip(axes[0], scenarios):
    sizes = [len(cols) for cols in imputer.imputation_order_.values()] or [0]
    bar_labels = list(imputer.imputation_order_.keys()) or ["-"]
    colors = [FREQ_COLORS.get(normalize_freq_label(l), "#aaaaaa") for l in bar_labels]
    ax.bar(range(len(sizes)), sizes, color=colors, edgecolor="white")
    ax.set_xticks(range(len(bar_labels)))
    ax.set_xticklabels(bar_labels)
    ax.set_ylabel("Colonnes imputees a l'etape")
    ax.set_title(title, fontweight="bold", fontsize=11)
fig.suptitle("Nombre de colonnes imputees par etape - 3 scenarios", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()